In [1]:
"""
Download a random subset of the English MultiEURLEX dataset and save it to the Data folder.
"""

import json
import os
import tarfile
import random
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from tqdm import tqdm

load_dotenv()
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    raise EnvironmentError(
        "Set HF_TOKEN in project root .env (see .env.example)"
    )

# Configuration
TAR_SPLIT_FILES = {"train.jsonl": "train.jsonl"}
LABEL_LEVEL = "level_1"
DOC_LIMIT = 1000

def main():
    data_dir = Path("Data")
    data_dir.mkdir(exist_ok=True, parents=True)

    # Use existing data if already downloaded
    out_path = data_dir / "train.jsonl"
    if out_path.exists():
        with open(out_path, encoding="utf-8") as f:
            count = sum(1 for _ in f)
        print(f"Data already exists at {out_path}. Using existing file ({count} documents).")
        return

    print("Downloading MultiEURLEX archive from Hugging Face...")
    tar_path = hf_hub_download(
        repo_id="coastalcph/multi_eurlex",
        filename="data/multi_eurlex.tar.gz",
        repo_type="dataset",
        token=HF_TOKEN
    )

    summary = {}
    with tarfile.open(tar_path, "r:gz") as tar:
        for member in tar.getmembers():
            base_name = member.name.split("/")[-1]
            if base_name not in TAR_SPLIT_FILES:
                continue

            print(f"Scanning {base_name} to sample {DOC_LIMIT} random English documents...")
            valid_indices = []
            f = tar.extractfile(member)
            if f is None: continue

            # First pass: find all English documents
            all_lines = f.readlines()
            for i, line in enumerate(all_lines):
                data = json.loads(line.decode("utf-8"))
                if isinstance(data.get("text"), dict) and "en" in data["text"]:
                    valid_indices.append(i)

            # Randomly sample indices
            sample_size = min(DOC_LIMIT, len(valid_indices))
            sampled_indices = set(random.sample(valid_indices, sample_size))

            out_path = data_dir / TAR_SPLIT_FILES[base_name]
            count = 0

            # Second pass: write sampled documents
            with open(out_path, "w", encoding="utf-8") as out_f:
                pbar = tqdm(desc=f"Writing {base_name}", total=sample_size, unit=" docs")
                for i, line in enumerate(all_lines):
                    if i in sampled_indices:
                        data = json.loads(line.decode("utf-8"))
                        record = {
                            "celex_id": data.get("celex_id", ""),
                            "text": data["text"]["en"],
                            "labels": data.get("eurovoc_concepts", {}).get(LABEL_LEVEL, data.get("labels", [])),
                        }
                        out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                        count += 1
                        pbar.update(1)
                pbar.close()

            summary[base_name.replace(".jsonl", "")] = count
            print(f"  {out_path.name}: {count} documents saved randomly.")

    summary_meta = {
        "dataset": "coastalcph/multi_eurlex",
        "language": "en",
        "label_level": LABEL_LEVEL,
        "splits": summary,
    }
    with open(data_dir / "dataset_info.json", "w", encoding="utf-8") as f:
        json.dump(summary_meta, f, indent=2)

    print("Random sampling and extraction complete.")

if __name__ == "__main__":
    main()

Data already exists at Data/train.jsonl. Using existing file (1000 documents).


/home/mwc/.conda/envs/Data/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd
import json

# Load the JSONL file into a DataFrame (skip empty lines to avoid parser errors)
file_path = "Data/train.jsonl"
with open(file_path, "r", encoding="utf-8") as f:
    df = pd.DataFrame([json.loads(line) for line in f if line.strip()])

# Display the first few rows and shape
print(f"DataFrame shape: {df.shape}")
display(df.head())

DataFrame shape: (1000, 3)


,celex_id,text,labels
0,31998D0057,COMMISSION DECISION of 28 November 1997 approv...,"[100152, 100144, 100161, 100156]"
1,31996R1262,COMMISSION REGULATION (EC) No 1262/96 of 1 Jul...,"[100147, 100145, 100157]"
2,32002D0924,Commission Decision\nof 23 July 1999\non the c...,"[100161, 100144, 100159]"
3,32001R2205,Commission Regulation (EC) No 2205/2001\nof 14...,"[100149, 100157, 100156]"
4,32002L0074,Directive 2002/74/EC of the European Parliamen...,"[100153, 100145, 100144, 100149]"


# CHUNKING

In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Assuming 'df' is your DataFrame with a 'text' column
# We will now split each text into paragraphs

# Initialize the text splitter for paragraph-based chunking
# It will try to split by double newline (paragraph), then single newline, then space
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,  # Max characters per chunk (as requested)
    chunk_overlap=0, # No overlap (as requested)
    separators=["\n\n", "\n", " ", ""], # Prioritize splitting by paragraphs and lines
    length_function=len,
    is_separator_regex=False,
)

all_chunks = []
for index, row in df.iterrows():
    # Create a single Document for the full text content
    full_doc = Document(page_content=row['text'])
    # Split this full document into smaller chunks based on the defined strategy
    chunks = text_splitter.split_documents([full_doc])
    all_chunks.extend(chunks)

# Update the 'documents' variable to hold the newly created chunks
documents = all_chunks

print(f"Created {len(documents)} LangChain document chunks.")
print("First few document chunks example:")
for i, doc in enumerate(documents[:3]): # Print first 3 chunks for verification
    print(f"Chunk {i}: {doc.page_content[:200]}...") # Show first 200 characters of content
    print(f"Metadata: {doc.metadata}") # Show metadata (which should be empty as per prior request)
    print("---")

Created 4000 LangChain document chunks.
First few document chunks example:
Chunk 0: COMMISSION DECISION of 28 November 1997 approving the programme for the eradication of brucella melitensis for 1998 presented by Spain and fixing the level of the Community's financial contribution (O...
Metadata: {}
---
Chunk 1: Whereas the measures provided for in this Decision are in accordance with the opinion of the Standing Veterinary Committee,
HAS ADOPTED THIS DECISION:
Article 1
The programme for the eradication of br...
Metadata: {}
---
Chunk 2: COMMISSION REGULATION (EC) No 1262/96 of 1 July 1996 amending Regulation (EEC) No 1059/83 on storage contracts for table wine, grape must, concentrated grape must and rectified concentrated grape must...
Metadata: {}
---


# EMBEDDINGS and VECTOR DB

In [4]:
# Reinstall necessary libraries, ensuring latest compatible versions
%pip install langchain langchain-community langchain-chroma chromadb ollama langchain-ollama > /dev/null

from pathlib import Path
from langchain_community.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma
from tqdm import tqdm

# Wrapper to show embedding progress with tqdm
class TqdmOllamaEmbeddings(OllamaEmbeddings):
    def embed_documents(self, texts):
        batch_size = 32
        all_embeddings = []
        for i in tqdm(range(0, len(texts), batch_size), desc="Embedding", unit="batch"):
            batch = texts[i : i + batch_size]
            all_embeddings.extend(super().embed_documents(batch))
        return all_embeddings

# Initialize Ollama Embeddings (ensure Ollama server is running with the 'llama2' model pulled)
# You might need to change 'llama2' to another model if you've pulled a different one.
print("Initializing Ollama Embeddings...")
embedding_model = TqdmOllamaEmbeddings(model="qwen3-embedding:0.6b")

# Persist Chroma DB in Data/ folder
PERSIST_DIR = "Data/chroma_db"
COLLECTION_NAME = "eurlex_documents"

# Load existing vector store if it already exists, otherwise create from documents
if Path(PERSIST_DIR).exists() and (Path(PERSIST_DIR) / "chroma.sqlite3").exists():
    print("Loading existing ChromaDB vector store from Data/chroma_db...")
    vectorstore = Chroma(
        persist_directory=PERSIST_DIR,
        embedding_function=embedding_model,
        collection_name=COLLECTION_NAME,
    )
    print("ChromaDB vector store loaded successfully!")
else:
    print("Creating ChromaDB vector store (this may take a while)...")
    Path(PERSIST_DIR).mkdir(parents=True, exist_ok=True)
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        collection_name=COLLECTION_NAME,
        persist_directory=PERSIST_DIR,
    )
    print("ChromaDB vector store created and saved successfully!")

print(f"Number of documents in vector store: {vectorstore._collection.count()}")

Note: you may need to restart the kernel to use updated packages.
Initializing Ollama Embeddings...
Loading existing ChromaDB vector store from Data/chroma_db...
ChromaDB vector store loaded successfully!
Number of documents in vector store: 4000


# RETRIEVER

In [5]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

# Initialize ChatOllama for structured system + user messages
# Ensure 'llama3.2' is pulled in your Ollama server
print("Initializing ChatOllama with llama3.2...")
chat_model = ChatOllama(model="llama3.2")
print("ChatOllama initialized.")

# Define a sample query (relevant to EU legislation in the corpus, e.g. tariff classification)
query = "what does the COUNCIL REGULATION (EC) No 1887/94 of 27 July 1994 fixing the basic price state?"
k = 3  # Number of documents to retrieve per search
PREVIEW_LEN = 200  # Characters to show per document snippet

# --- Output formatting helpers ---
SEP = "=" * 72
SUB = "-" * 72

def print_section(title: str, subtitle: str = ""):
    print(f"\n{SEP}")
    print(f"  {title}")
    if subtitle:
        print(f"  {subtitle}")
    print(SEP)

def print_doc(i: int, n: int, content: str, score: float = None):
    print(f"\n  [{i}/{n}]", end="")
    if score is not None:
        print(f"  (similarity score: {score:.4f})")
    else:
        print()
    snippet = content[:PREVIEW_LEN].replace("\n", " ").strip()
    print(f"  {snippet}...")

# --- Header ---
print_section("RETRIEVAL DEMO", "Comparing three retrieval methods on the vector store")
print(f"\n  Query:  {query}")
print(f"  k =     {k} (documents per retrieval)")

# --- Retrieval Method 1: Similarity Search ---
print_section("Method 1: Similarity Search", "vectorstore.similarity_search(query, k=k)")
similarity_docs = vectorstore.similarity_search(query, k=k)
print(f"\n  Retrieved: {len(similarity_docs)} document(s)")
for i, doc in enumerate(similarity_docs, 1):
    print_doc(i, len(similarity_docs), doc.page_content)

# --- Retrieval Method 2: Similarity Search with Score ---
print_section("Method 2: Similarity Search with Score", "vectorstore.similarity_search_with_score(query, k=k)")
scored_docs = vectorstore.similarity_search_with_score(query, k=k)
print(f"\n  Retrieved: {len(scored_docs)} document(s)")
for i, (doc, score) in enumerate(scored_docs, 1):
    print_doc(i, len(scored_docs), doc.page_content, score=score)

# --- Retrieval Method 3: Search by Vector ---
print_section("Method 3: Search by Vector", "vectorstore.similarity_search_by_vector(embedding, k=k)")
query_embedding = embedding_model.embed_query(query)
vector_search_docs = vectorstore.similarity_search_by_vector(query_embedding, k=k)
print(f"\n  Retrieved: {len(vector_search_docs)} document(s)")
for i, doc in enumerate(vector_search_docs, 1):
    print_doc(i, len(vector_search_docs), doc.page_content)

# --- Footer ---
print(f"\n{SUB}")
print("  All three retrieval methods completed.")
print(SUB)

# --- RAG: Generate answer from retrieved context via Ollama ---
SYSTEM_PROMPT = """You are a question-answering assistant for EU legislation documents.
Your task is to answer the user's question using ONLY the retrieved context provided below.

RULES:
- Answer ONLY based on the retrieved context. Do not use external knowledge.
- If the answer cannot be found in the context, respond: "The answer is not provided in the retrieved context."
- Be concise. When possible, cite or reference the relevant passage from the context.
- Do not speculate or make up information."""

def build_rag_prompt(query: str, docs: list) -> str:
    context_parts = []
    for i, doc in enumerate(docs, 1):
        context_parts.append(f"--- Document {i} ---\n{doc.page_content}")
    context = "\n\n".join(context_parts)
    return f"""USER QUESTION:
{query}

RETRIEVED CONTEXT:
{context}"""

print_section("RAG: Answer from Ollama (llama3.2)", "Using similarity_search results as context")
messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=build_rag_prompt(query, similarity_docs)),
]
response = chat_model.invoke(messages)
print(f"\n  Answer:\n  {response.content}")



Initializing ChatOllama with llama3.2...
ChatOllama initialized.

  RETRIEVAL DEMO
  Comparing three retrieval methods on the vector store

  Query:  what does the COUNCIL REGULATION (EC) No 1887/94 of 27 July 1994 fixing the basic price state?
  k =     3 (documents per retrieval)

  Method 1: Similarity Search
  vectorstore.similarity_search(query, k=k)

  Retrieved: 3 document(s)

  [1/3]
  HAS ADOPTED THIS REGULATION: Article 1 The rates of the refunds applicable to the basic products listed in Annex I to Regulation (EC) No 1043/2005 and in Article 1(1) of Regulation (EEC) No 2771/75, a...

  [2/3]
  COMMISSION REGULATION (EEC) No 2850/93 of 19 October 1993 laying down the prices and amounts fixed in ecus in the olive oil sector and reduced as a result of monetary realignments in the 1992/93 marke...

  [3/3]
  COMMISSION REGULATION (EC) No 154/2007 of 16 February 2007 fixing the minimum selling prices for butter for the 25th individual invitation to tender under the standing invit